# Week 53, Day 1: Spotting DP problems

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w053d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
calls = [0]

def ways(n):
    calls[0] += 1
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

print(ways(20), calls[0])

memo = {}
calls[0] = 0
def ways_memo(n):
    calls[0] += 1
    if n <= 1:
        return 1
    if n not in memo:
        memo[n] = ways_memo(n - 1) + ways_memo(n - 2)
    return memo[n]

print(ways_memo(20), calls[0])

In [ ]:
from functools import cache

coins = (1, 3, 4)

@cache
def fewest(amount):
    if amount == 0:
        return 0
    options = [fewest(amount - c) for c in coins if c <= amount]
    return 1 + min(options)

print([fewest(a) for a in range(11)])   # greedy gives 3 for 6; this gives 2

In [ ]:
from functools import cache

@cache
def strings(n, last_was_one):
    """Binary strings of length n with no two 1s side by side."""
    if n == 0:
        return 1
    total = strings(n - 1, False)              # put a 0
    if not last_was_one:
        total += strings(n - 1, True)           # put a 1
    return total

print([strings(n, False) for n in range(1, 9)])

In [ ]:
from functools import cache

def cheapest_climb(cost):
    n = len(cost)

    @cache
    def f(i):                                   # cheapest cost to stand on step i
        if i <= 1:
            return 0
        return min(f(i - 1) + cost[i - 1], f(i - 2) + cost[i - 2])

    return f(n)

print(cheapest_climb([10, 15, 20]), cheapest_climb([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

In [ ]:
from functools import cache

def biggest_path(triangle):
    """Largest total from the top to the bottom, moving to either number below."""
    @cache
    def best(r, c):
        if r == len(triangle) - 1:
            return triangle[r][c]
        return triangle[r][c] + max(best(r + 1, c), best(r + 1, c + 1))
    return best(0, 0)

print(biggest_path([[3], [7, 4], [2, 4, 6], [8, 5, 9, 3]]))

> **Key idea.** Dynamic programming solves problems with optimal substructure and overlapping subproblems by solving each distinct subproblem once. Design the state as exactly what the rest of the answer depends on: too little gives wrong answers, too much leaves nothing to share. Write the recurrence first: define the function in a sentence, give base cases, describe the choice and what each option leaves, and name the state that answers the question. Translate it to recursion with functools.cache, defined inside the outer function and with hashable arguments. The running time is the number of states times the work per state.

### Exercise: Fewest squares

Write `fewest_squares(n)`: the fewest perfect squares (1, 4, 9, 16, ...) that add up to n, for n ≥ 0 (0 needs none). State: the amount left. Choice: which square to use next. Use `functools.cache` inside the function.

In [ ]:
from functools import cache

def fewest_squares(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_squares():\n    \"fewest squares\"\n    assert [fewest_squares(n) for n in range(14)] == [0, 1, 2, 3, 1, 2, 3, 4, 2, 1, 2, 3, 3, 2]\n    assert fewest_squares(12) == 3 and fewest_squares(13) == 2 and fewest_squares(43) == 3\n    assert fewest_squares(500) == 2 and fewest_squares(247) == 4")

### Exercise: A frog on stones

A frog starts on stone 0 and wants stone n − 1. From stone i it can jump to i + 1 or i + 2, and a jump costs the difference in height, `abs(h[i] - h[j])`. Write `frog_cost(heights)`: the cheapest total. Write the recurrence first: f(i) is the cheapest way to reach stone i.

In [ ]:
from functools import cache

def frog_cost(heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef brute(h):\n    n = len(h)\n    best = [float(\"inf\")]\n    def go(i, total):\n        if i == n - 1:\n            best[0] = min(best[0], total)\n        for j in (i + 1, i + 2):\n            if j < n:\n                go(j, total + abs(h[i] - h[j]))\n    go(0, 0)\n    return best[0]\ndef test_frog():\n    \"frog\"\n    assert frog_cost([10, 30, 40, 20]) == 30 and frog_cost([10, 10]) == 0 and frog_cost([5]) == 0\n    assert frog_cost([30, 10, 60, 10, 60, 50]) == 40\n    rng = random.Random(1)\n    for _ in range(100):\n        h = [rng.randint(0, 50) for _ in range(rng.randint(1, 12))]\n        assert frog_cost(h) == brute(h)")

### Exercise: No long streaks

Count the strings of length n made of the letters `"W"` and `"L"` (a season of wins and losses) with no more than k equal letters in a row. Write `seasons(n, k)`. The state must include the **current streak length**. Then `seasons_with(n, k, first)`: the same count, but only strings starting with `first`.

In [ ]:
from functools import cache

def seasons(n, k):
    ___

def seasons_with(n, k, first):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef brute(n, k, first=None):\n    count = 0\n    for s in itertools.product(\"WL\", repeat=n):\n        if first and (not s or s[0] != first):\n            continue\n        run, ok = 0, True\n        for i, ch in enumerate(s):\n            run = run + 1 if i and s[i - 1] == ch else 1\n            if run > k:\n                ok = False\n                break\n        count += ok\n    return count\ndef test_seasons():\n    \"streaks\"\n    assert seasons(3, 1) == 2 and seasons(3, 2) == 6 and seasons(0, 2) == 1\n    for n in range(0, 11):\n        for k in range(1, 5):\n            assert seasons(n, k) == brute(n, k), (n, k)\n            if n:\n                assert seasons_with(n, k, \"W\") == brute(n, k, \"W\")\n    assert seasons(60, 3) == 2 * seasons_with(60, 3, \"L\")")

### Exercise: Collections, not orders

Write `orders(amount, coins)`: how many **sequences** of coins add up to the amount (1 + 2 and 2 + 1 count separately), and `collections(amount, coins)`: how many **collections** do (they count once). Coins are distinct positive whole numbers, with unlimited supply. For collections, the state is `(amount_left, i)`: you may only use `coins[i:]`, so each collection is built in one fixed order.

In [ ]:
from functools import cache

def orders(amount, coins):
    ___

def collections(amount, coins):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_counts():\n    \"orders and collections\"\n    assert orders(4, [1, 2]) == 5 and collections(4, [1, 2]) == 3\n    assert orders(0, [1, 2]) == 1 and collections(0, [5]) == 1 and collections(3, [2]) == 0\n    assert collections(200, [1, 2, 5, 10, 20, 50, 100, 200]) == 73682\n    assert collections(10, [2, 5, 3, 6]) == 5 and orders(10, [2, 5, 3, 6]) == 17\n    for amount in range(0, 13):\n        for coins in ([1, 3], [2, 3, 7], [1, 2, 5]):\n            seqs = sum(1 for r in range(amount + 1) for s in itertools.product(coins, repeat=r) if sum(s) == amount)\n            sets = len({tuple(sorted(s)) for r in range(amount + 1) for s in itertools.product(coins, repeat=r) if sum(s) == amount})\n            assert orders(amount, coins) == seqs and collections(amount, coins) == sets")

### Exercise: The best path down

Write `best_path(triangle)`: `(total, path)`, the largest total from the top of a number triangle to its bottom row (from position c you may move to c or c + 1 in the row below), and the list of numbers on that path. On a tie, prefer the left move. Then `count_best(triangle)`: how many top-to-bottom paths achieve that total.

In [ ]:
from functools import cache

def best_path(triangle):
    ___

def count_best(triangle):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef paths(tri):\n    out = []\n    def go(r, c, acc):\n        acc = acc + [tri[r][c]]\n        if r == len(tri) - 1:\n            out.append(acc)\n            return\n        go(r + 1, c, acc)\n        go(r + 1, c + 1, acc)\n    go(0, 0, [])\n    return out\ndef test_triangle():\n    \"paths\"\n    tri = [[3], [7, 4], [2, 4, 6], [8, 5, 9, 3]]\n    assert best_path(tri) == (23, [3, 7, 4, 9]) and count_best(tri) == 1\n    assert best_path([[5]]) == (5, [5]) and count_best([[1], [1, 1]]) == 2\n    rng = random.Random(1)\n    for _ in range(60):\n        n = rng.randint(1, 7)\n        tri = [[rng.randint(0, 4) for _ in range(r + 1)] for r in range(n)]\n        ps = paths(tri)\n        top = max(sum(p) for p in ps)\n        total, path = best_path(tri)\n        assert total == top and sum(path) == top and path in ps\n        assert count_best(tri) == sum(1 for p in ps if sum(p) == top)\n        assert path == max(ps, key=sum)\n    big = [[(r * 7 + c * 13) % 100 for c in range(r + 1)] for r in range(100)]\n    row = big[-1][:]\n    for r in range(98, -1, -1):\n        row = [big[r][c] + max(row[c], row[c + 1]) for c in range(r + 1)]\n    total, path = best_path(big)\n    assert total == row[0] and sum(path) == total and len(path) == 100")

### Bug hunt: Zero ways

This counts the ways to climb n stairs one or two at a time, but its base case says there are **no** ways to climb zero stairs, so every count collapses to `0`. There's exactly one way to climb nothing: do nothing. Fix the base case so it prints `[1, 1, 2, 3, 5, 8]`.

In [ ]:
from functools import cache

@cache
def ways(n):
    if n < 0:
        return 0
    if n == 0:
        return 0
    return ways(n - 1) + ways(n - 2)

print([ways(n) for n in range(6)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one way to do nothing\"\n    expect_output(\"[1, 1, 2, 3, 5, 8]\")")

### Bug hunt: A cache that outlives its input

The cached function sits at module level and reads the list `cost` from outside, so the second call reuses answers worked out for the **first** list and prints `[2, 2]`. Move the cached function inside `cheapest`, so each list gets its own cache and it prints `[2, 6]`.

In [ ]:
from functools import cache

cost = []

@cache
def f(i):
    if i <= 1:
        return 0
    return min(f(i - 1) + cost[i - 1], f(i - 2) + cost[i - 2])

def cheapest(costs):
    cost[:] = costs
    return f(len(costs))

print([cheapest([1, 2, 1, 5]), cheapest([3, 3, 3, 3])])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one cache per input\"\n    expect_output(\"[2, 6]\")")

### Bug hunt: A state that forgets

This counts binary strings of length 4 with no two 1s side by side, but the state is only the length left, so the memo can't tell "after a 0" from "after a 1": it remembers whichever came first and prints `16` (every string of length 4!) instead of `8`. Add whether the last digit was a 1 to the state (and to the memo key).

In [ ]:
memo = {}

def strings(n, last_was_one):
    if n == 0:
        return 1
    if n in memo:
        return memo[n]
    total = strings(n - 1, False)
    if not last_was_one:
        total += strings(n - 1, True)
    memo[n] = total
    return total

print(strings(4, False))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the whole state\"\n    expect_output(\"8\")")

### Puzzle: Memoised triangle

Arrange the lines into the memoised best path down a number triangle. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of states in the fewest-coins DP for amount 10 (amounts 0 to 10)
q = ___   # strings(3, False): binary strings of length 3 with no two 1s together
r = ___   # True or False: merge sort has overlapping subproblems
s = ___   # the number of ways to climb 5 stairs one or two at a time
t = ___   # the number of (r, c) states in a triangle with 4 rows

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 11 and q == 5 and r is False, \"p, q and r\"\n    assert s == 8 and t == 10, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: tiling a wall

A wall `rows` tiles high and `n` tiles long is covered with 1 × 2 dominoes, each laid flat or upright. Write `tilings(rows, n)`: the number of ways, for 1 ≤ rows ≤ 5 and n ≥ 0, by **column-by-column DP**:

- Fill the wall one column at a time. The state is `(column, mask)`, where `mask` is a bitmask of the cells in this column already covered by flat dominoes poking in from the left.
- Within a column, fill the cells top to bottom: a cell already covered is skipped; otherwise put an upright domino (if the cell below is free) or a flat domino that covers this cell and the same row of the **next** column (setting that bit in the next mask).
- The answer is the state after the last column with nothing poking out: mask 0.

Then `tilings_3(n)`: the 3-row count from your DP, and `check_3(n)`: True if it matches the recurrence T(n) = 4T(n − 2) − T(n − 4) for even n ≥ 4 (with T(0) = 1, T(2) = 3). This "broken profile" method is the standard way to count tilings of narrow boards.

In [ ]:
from functools import cache

def tilings(rows, n):
    ___

def tilings_3(n):
    ___

def check_3(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tilings():\n    \"tilings\"\n    assert [tilings(2, n) for n in range(8)] == [1, 1, 2, 3, 5, 8, 13, 21]\n    assert [tilings(3, n) for n in range(9)] == [1, 0, 3, 0, 11, 0, 41, 0, 153]\n    assert [tilings(4, n) for n in range(7)] == [1, 1, 5, 11, 36, 95, 281]\n    assert [tilings(1, n) for n in range(6)] == [1, 0, 1, 0, 1, 0]\n    assert tilings(5, 4) == 95 and tilings(5, 6) == 1183 and tilings(3, 0) == 1\n    assert tilings_3(30) == 299303201 and all(check_3(n) for n in range(4, 31, 2))\n    assert tilings(5, 8) == 14824 and tilings(5, 10) == 185921")

## Warm-up

The number of ways to climb 20 stairs taking one or two at a time. The plain recursion asks for `ways(18)` twice, `ways(17)` three times, `ways(16)` five times, and so on: over twenty thousand calls for 21 different questions. The memo answers each question once.

## Two tell-tale signs

A problem suits dynamic programming when it has both:

- **Optimal substructure**: the best (or total) answer is built from the answers to smaller versions of the same problem. The cheapest route to the top stair goes through the cheapest route to one of the stairs just below it.
- **Overlapping subproblems**: those smaller versions come up **again and again** in the search. If they never repeat (as in merge sort, where each half is different), plain recursion is already efficient, and memoising wastes memory.

The phrases that suggest it: "how many ways", "the fewest", "the most", "the longest", "the cheapest", "is it possible", over a sequence, a grid, a budget or a set of choices that come **in an order**. Greedy (last week) also needs optimal substructure, but commits to one choice; DP tries **every** choice at each step and keeps the best, which is why it works when greed doesn't.

## Designing the state

The **state** is the information a subproblem needs: enough to decide everything that can happen from here, and no more. Ask: "if I stop halfway, what must I know to finish optimally?"

- Climbing stairs: only **which stair** you're on. How you got there doesn't matter.
- Fewest coins: only the **amount left**.
- Binary strings with no two 1s together: the **length left** and **whether the last digit was a 1**, because that decides what comes next. Leave out the last digit and the answers are wrong; that was Friday's bug, remembering too little.
- Counting coin **collections** (where 1 + 2 and 2 + 1 are the same): the amount left **and which coins you may still use**, so each collection is built in one fixed order.

Too little in the state gives wrong answers. Too much (the whole path so far) makes every state different, so nothing overlaps and DP becomes plain backtracking.

## The recurrence first

Before code, write the **recurrence** in words or symbols:

1. **Define** the function in one sentence: "f(i) = the cheapest cost to reach stair i".
2. **Base cases**: the smallest states, answered directly: "f(0) = f(1) = 0, you can start on either".
3. **The choice**: the last (or first) decision, and what each option leaves: "you arrived from stair i − 1 or i − 2, paying that stair's cost: f(i) = min(f(i − 1) + cost[i − 1], f(i − 2) + cost[i − 2])".
4. **The answer**: which state is the question? "f(n), the top, just past the last stair".

Check it by hand on a tiny input before coding. Most DP bugs are a wrong sentence in step 1 or a missing base case in step 2, not a typing error.

## From recurrence to code

With the recurrence written, the code is a direct translation, and `functools.cache` (Week 16) does the remembering. Three practical points:

- Arguments must be **hashable**: pass positions (indices) into a list rather than slices, and tuples rather than lists.
- Define the cached function **inside** the outer function, as above, so each call gets a fresh cache for its own input. A cache at module level remembers answers for the **previous** input too.
- Deep recursion hits Python's limit (about 1000 calls deep). For long inputs, tomorrow's **tabulation** fills a table in a loop instead, with no recursion at all.

That puzzle is adapted from Project Euler problem 18, where a 100-row version (problem 67) has more routes than could ever be checked one by one, but only 5,050 states.

## How fast is it?

The running time of a memoised DP is

**number of states × work per state** (the work outside the recursive calls).

- Stairs: n states, O(1) each, so O(n).
- Fewest coins: amount + 1 states, one option per coin: O(amount × coins).
- The triangle: about n²/2 states, O(1) each: O(n²).
- Strings with no two 1s: 2n states, O(1) each: O(n).

Memory is the number of states (plus the recursion depth). This count is also the quickest check that your state is sensible: if it's exponential, you've put too much in it.

## Practice

## Quiz

## Daily challenge

## Before you go

Write down, in one sentence each, the state for two problems from today, and one fact you deliberately left **out** of each state because the rest of the answer doesn't depend on it. Tomorrow: the same recurrences as tables filled by loops, with no recursion and often far less memory.